In [1]:
from collections import defaultdict
import pandas as pd
import json
import os
import re
import hashlib
import csv
from psycopg2 import sql
from psycopg2.extras import execute_batch
import sys
sys.path.append("..")
from match_url_checker import *
from db_operations import get_db_conn
from dotenv import load_dotenv
load_dotenv("../../src/config/.env")

True

In [2]:
def filter_dataframe(df):
    valid_qp = {"0", "Yes"} | set(map(str, range(1, 6)))
    qp_clean = df["Query Project"].astype(str).str.strip()
    df_filtered = df[qp_clean.isin(valid_qp)]
    return df_filtered.copy()

In [3]:
# Folder path
folder = "../results/data"

# Get list of all CSV files in the folder
files = [f for f in os.listdir(folder) if f.endswith(".csv")]

# Read and concat
df = pd.concat([pd.read_csv(os.path.join(folder, f)) for f in files], ignore_index=True)

print(df.shape)

(1056081, 11)


In [4]:
df = df[df["project_id"] >= 0]
df = df[df["version"] >= 0]

In [5]:
df=df.drop(["source_project", "source_project_version"], axis=1)

In [6]:
df.head()

,hash,project_id,version,license,method_name,file_location,repository_url,query_project,violation
0,38cc1c786257364d273998bdebed06b2,2479146364,1.561335e+12,-,weight_op,modules/layers.py:267,https://github.com/tanglang96/MDENAS/blob/ee6c...,3,NaN
1,106fb9dbfa3e227fd12d41dcb7c7b7c1,1077307302,1.681413e+12,Other,space_indices,./i2sb/util.py:86,https://github.com/NVlabs/I2SB/blob/1ffdfaaf05...,3,NaN
2,b64f162bc4381258da1f40f31bf41773,1077307302,1.681413e+12,Other,__init__,./logger.py:36,https://github.com/NVlabs/I2SB/blob/1ffdfaaf05...,3,NaN
3,032ffe9d3bf14f43077ccb5adcd92006,3738289328,1.691649e+12,Apache License 2.0,cupsdCopyLocation,./scheduler/auth.c:1282,https://github.com/microsoft/cups/blob/e59cc13...,0,NaN
4,0b972bdbf783aa0c225c47b086e4c7d8,2330826691,1.720756e+12,Apache License 2.0,main,./systemv/lpmove.c:30,https://github.com/apple/cups/blob/master/syst...,Yes,Apache-2.0 same license Apache-2.0


In [7]:
# df.to_csv("../results/data/combined_all_methods.csv", index=False)

In [8]:
def detect_language(path: str) -> str:
    # Remove any trailing line number (e.g. file.py:123 -> file.py)
    clean_path = re.sub(r":\d+$", "", str(path))
    
    ext = os.path.splitext(clean_path)[1].lower()
    
    mapping = {
        ".c": "c", ".h": "c",
        ".cpp": "cpp", ".cc": "cpp", ".hpp": "cpp",
        ".cs": "cs", ".java": "java",
        ".js": "js", ".ts": "js",
        ".py": "py",
    }
    return mapping.get(ext, "other")

In [9]:
df["Language"] = df["file_location"].apply(detect_language)

In [10]:
def filter_trivial_functions_by_name(df, file_col="file_location"):
    if df.empty:
        return df.copy()

    if file_col not in df.columns:
        raise KeyError(f"Column '{file_col}' not found in DataFrame")

    df = df.copy()

    # --- Filter by Method Name format ---
    df = df[df['method_name'].str.match(r'^[A-Za-z_][A-Za-z0-9_]{2,}$', na=False)]

    # --- Filter by File Location format ---
    df = df[df[file_col].str.contains(r'\w+/\w+.*\.\w+', na=False)]

    if df.empty:
        df["is_trivial"] = False
        return df

    # --- Trivial patterns ---
    trivial_patterns = {
        "c":    [r'^(get|set|init|reset|free|alloc|load|save|open|close|input|output|error|message|complete)$', r'^(main)$'],
        "cpp":  [r'^(get|set|init|reset|copy|assign|release|load|save|open|close|input|output|error|message|complete)$', r'^(main|operator.*)$'],
        "cs":   [r'^(get|set|reset|dispose|clone|load|save|open|close|input|output|error|message|complete|is[A-Z][A-Za-z0-9_]*)$', r'^(Main)$'],
        "java": [r'^(get|set|load|save|open|close|input|output|error|message|complete|is[A-Z][A-Za-z0-9_]*|clone|toString|hashCode|equals)$', r'^(main)$'],
        "js":   [r'^(get|set|reset|constructor|load|save|open|close|input|output|error|message|complete)$', r'^(main)$'],
        "py":   [r'^__.*__$', r'^(init|get|set|reset|load|save|open|close|input|output|error|message|complete|is_[a-z0-9_]+)$', r'^(main)$'],
        "other":[r'^__.*__$', r'^(init|get|set|reset|load|save|open|close|input|output|error|message|complete|is_[a-z0-9_]+)$', r'^(main)$'],
    }

    compiled_patterns = {
        lang: re.compile("|".join(pats), re.IGNORECASE)
        for lang, pats in trivial_patterns.items()
    }

    # --- Flagging instead of filtering ---
    def is_trivial(name, lang):
        regex = compiled_patterns.get(lang)
        return bool(regex and regex.match(str(name)))

    df["is_trivial"] = df.apply(
        lambda row: is_trivial(row["method_name"], row["Language"]),
        axis=1
    )

    return df.reset_index(drop=True)


In [11]:
df1 = filter_trivial_functions_by_name(df)

In [12]:
df1.head()

,hash,project_id,version,license,method_name,file_location,repository_url,query_project,violation,Language,is_trivial
0,38cc1c786257364d273998bdebed06b2,2479146364,1.561335e+12,-,weight_op,modules/layers.py:267,https://github.com/tanglang96/MDENAS/blob/ee6c...,3,NaN,py,False
1,106fb9dbfa3e227fd12d41dcb7c7b7c1,1077307302,1.681413e+12,Other,space_indices,./i2sb/util.py:86,https://github.com/NVlabs/I2SB/blob/1ffdfaaf05...,3,NaN,py,False
2,032ffe9d3bf14f43077ccb5adcd92006,3738289328,1.691649e+12,Apache License 2.0,cupsdCopyLocation,./scheduler/auth.c:1282,https://github.com/microsoft/cups/blob/e59cc13...,0,NaN,c,False
3,0b972bdbf783aa0c225c47b086e4c7d8,2330826691,1.720756e+12,Apache License 2.0,main,./systemv/lpmove.c:30,https://github.com/apple/cups/blob/master/syst...,Yes,Apache-2.0 same license Apache-2.0,c,True
4,0be14c6a4fa1f158732936c8fbe570de,3738289328,1.691649e+12,Apache License 2.0,add_printer_filters,./scheduler/cupsfilter.c:693,https://github.com/microsoft/cups/blob/e59cc13...,0,NaN,c,False


In [13]:
df1.is_trivial.value_counts()

is_trivial
False    579540
True      44691
Name: count, dtype: int64

In [14]:
#from global_trivial_methods import load_global_trivial_names
#trivial_names = load_global_trivial_names(file_path="../input_files/global_trivial_names.json", threshold=30)

In [15]:
from global_trivial_methods import load_global_trivial_names

def filter_with_global_trivial_names(df, file_path="global_trivial_names.json", threshold=20):
    """
    Update dataframe with trivial flag based on global trivial method names.
    If a row is not already trivial, mark it True if it matches global trivial names.
    """
    trivial_names = load_global_trivial_names(file_path, threshold)

    df = df.copy()

    # Ensure the column exists
    if "is_trivial" not in df.columns:
        df["is_trivial"] = False

    # Mark as trivial if method is in trivial_names and not already trivial
    df.loc[(~df["is_trivial"]) & (df["method_name"].isin(trivial_names)), "is_trivial"] = True

    return df.reset_index(drop=True)


In [16]:
df2 = filter_with_global_trivial_names(df1)

In [17]:
df2.is_trivial.value_counts()

is_trivial
False    579540
True      44691
Name: count, dtype: int64

In [ ]:
df2 = df2[df2["is_trivial"]==False]

In [19]:
df2.shape

(579540, 11)

In [20]:
# Sort by version descending, so the latest version comes first
df_sorted = df2.sort_values(by=["hash", "project_id", "version"])

df_result = (
    df_sorted.groupby("hash")                    # group only by hash
    .filter(lambda g: len(g) > 1)                # keep only duplicates
    .groupby("hash")
    .apply(lambda g: g.iloc[[0, -1]])            # take first & last row
    .reset_index(drop=True)
)

/tmp/ipykernel_66760/2692500397.py:8: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.iloc[[0, -1]])            # take first & last row


In [21]:
df_result.shape

(395364, 11)

In [22]:
df_result.is_trivial.value_counts()

is_trivial
False    395364
Name: count, dtype: int64

In [28]:
df_result.columns

Index(['hash', 'project_id', 'version', 'license', 'method_name',
       'file_location', 'repository_url', 'query_project', 'violation'],
      dtype='object')

In [24]:
df_result.drop(columns=["is_trivial", "Language"], inplace=True)

In [25]:
df_result.project_id.dtype

dtype('int64')

In [26]:
df_result.shape

(395364, 9)

In [ ]:
#df_result.to_csv("../results/processed_data/clone_reused_code_repos.csv", index=False)

In [4]:
import pandas as pd
import psycopg2

def import_csv_to_table(csv_file, table_name, drop_if_exists=True):
    """Read CSV with pandas and insert into PostgreSQL."""
    conn = get_db_conn()
    cur = conn.cursor()

    if drop_if_exists:
        cur.execute(f"DROP TABLE IF EXISTS {table_name}")

    # Create table
    cur.execute(f"""
        CREATE TABLE IF NOT EXISTS {table_name} (
            hash VARCHAR(100),
            project_id VARCHAR(100),
            version VARCHAR(100),
            license TEXT,
            method_name TEXT,
            file_location TEXT,
            repository_url TEXT,
            query_project TEXT,
            violation TEXT,
            granular_level_reached INT,
            code_content TEXT,
            similarity NUMERIC(4,3)
        )
    """)

    # Read CSV with pandas
    df = pd.read_csv(csv_file, encoding='utf-8',  nrows=300_000)  # can also use 'utf-8-sig' if BOM exists
    df = df[df['query_project'] != "Yes"]

    # Ensure all required columns exist
    required_cols = [
        "hash", "project_id", "version", "license", "method_name",
        "file_location", "repository_url", "query_project"
    ]
    for col in required_cols:
        if col not in df.columns:
            df[col] = None  # fill missing columns with None

    df['version'] = df['version'].astype(str).str.replace('.0', '', regex=False)
    # Add default columns
    df["granular_level_reached"] = -1
    df["code_content"] = ""
    df["similarity"] = None

    insert_columns = [
        "hash", "project_id", "version", "license", "method_name",
        "file_location", "repository_url", "query_project",
        "granular_level_reached", "code_content", "similarity"
    ]

    # Insert all rows in a batch
    values = [tuple(x) for x in df[insert_columns].to_numpy()]
    placeholders = ", ".join(["%s"] * len(insert_columns))
    insert_query = f"INSERT INTO {table_name} ({', '.join(insert_columns)}) VALUES ({placeholders})"
    execute_batch(cur, insert_query, values, page_size=1000)
    #cur.executemany(insert_query, values)

    conn.commit()
    cur.close()
    conn.close()
    print(f"CSV imported into table '{table_name}' successfully.")


In [6]:
def update_granular_levels(TABLE_NAME):
    conn = get_db_conn()
    cur = conn.cursor()

    # --- Fetch rows from table ---
    cur.execute(f"SELECT hash, project_id, version, license, method_name, file_location, repository_url FROM {TABLE_NAME} WHERE granular_level_reached < 0 LIMIT 200")
    rows = cur.fetchall()

    for i, row in enumerate(rows, 1):
        try:
            print(f"Processing row {i}...")
            hash_val, project_id, version, license_, method_name, file_location, repo_url = row

            # Default values
            granular_level_reached = 0

            # Verification steps
            if github_repo_exists(repo_url):
                granular_level_reached = 1
                commit_sha, _, _ = check_version_and_get_sha(repo_url, version)
                if commit_sha:
                    granular_level_reached = 2
                    if find_file_at_exact_path(repo_url):
                        granular_level_reached = 3
                        if find_method_in_file(repo_url, method_name):
                            granular_level_reached = 4

            # --- Update DB row ---
            update_query = f"""
                UPDATE {TABLE_NAME}
                SET granular_level_reached = %s
                WHERE hash = %s AND project_id = %s AND version = %s
            """
            cur.execute(update_query, (granular_level_reached, hash_val, project_id, version))
        except Exception as e:
            print(f"Error processing row {i}: {e}")
            continue

    conn.commit()
    cur.close()
    conn.close()
    print("Granular levels updated for all rows.")

In [5]:
INPUT_CSV_FILE = '../../results/processed_data/clone_reused_code_repos.csv'
#'../results/processed_data/others/verifier_demo.csv'
#"../results/processed_data/first_10_rows_test.csv"
#'../results/processed_data/clone_reused_code_repos.csv'
TABLE_NAME = "repository_data_source_verifier"  # change if needed
DROP_IF_EXISTS = True

In [6]:
import_csv_to_table(INPUT_CSV_FILE, TABLE_NAME, DROP_IF_EXISTS)

CSV imported into table 'repository_data_source_verifier' successfully.


In [7]:
update_granular_levels(TABLE_NAME)

Processing row 1...
--> Found version for timestamp 1712071726000 in microsoft/BotFramework-FunctionalTests. Commit SHA: e6ab44a37eeacf46d19569544cc392e9780b5e61
Processing row 2...
--> Found version for timestamp 1744665362000 in microsoft/BotBuilder-Samples. Commit SHA: e293d250d9a36fb6d789af67e67e099651be579d
Processing row 3...
--> Found version for timestamp 1441370306000 in Mbed-TLS/mbedtls. Commit SHA: 0a0c22e0efcf2f8f71d7e16712f80b8f77326f72
Processing row 4...
--> Found version for timestamp 1462944485000 in intel/incubator-mynewt-core. Commit SHA: dcb21d9048bc0c5f20d7690a8e42b63fe3c9d105
Processing row 5...
--> Found version for timestamp 1574298060000 in microsoft/MapsSDK-Unity. Commit SHA: 8a20073540c379a800c983e074fa8c1ef2617adb
Processing row 6...
--> Found version for timestamp 1594790870000 in microsoft/MRDL_Unity_Surfaces. Commit SHA: dc3f88b36c7df76f9460280199f551d525028b52
Processing row 7...
--> Found version for timestamp 1722637499000 in oneapi-src/oneDNN. Commit 

In [ ]:
import pandas as pd

output_csv = "../results/processed_data/first_10_rows_test.csv"

# Read CSV
df = pd.read_csv(INPUT_CSV_FILE)

# Select first 10 rows
df_first_10 = df.head(20)

# Write to new CSV
df_first_10.to_csv(output_csv, index=False)

print(f"First 10 rows written to '{output_csv}'")

First 10 rows written to '../results/processed_data/first_10_rows_test.csv'


In [9]:
import pandas as pd

INPUT_CSV_FILE = "../../results/Samsung_mTower_matches.csv"
output_csv = "../../results/processed_data/Samsung_mTower_matches.csv"
# Read CSV
df = pd.read_csv(INPUT_CSV_FILE)

df = df[df['query_project'] != "5"]

df = df.sort_values(by=["hash", "version", "project_id"])

# Write to new CSV
df.to_csv(output_csv, index=False)

print(f"Written to '{output_csv}'")

Written to '../../results/processed_data/Samsung_mTower_matches.csv'
